# Building Multi-Step Tool-Calling Datasets with Data Designer

Generate synthetic training data for agentic Reinforcement-Learning using NVIDIA **Data Designer** to enhance multi-step tool calling ability.

## Prerequisites

- **NVIDIA API Key** from [build.nvidia.com](https://build.nvidia.com) to access a remote LLM for generation. To create one, go to [API Keys](https://build.nvidia.com/settings/api-keys), sign in, create a key, and keep it ready for the configuration step below. Alternatively, you may choose to use your own endpoint or deployment.
- **Python 3.11+**
- **Tool definition files** in the `tools/` directory (included in this repo)
- Packages: `data-designer`, `pydantic`, `pandas`

## Objectives

By the end of this notebook, you will:

- Load known **tool schemas** as the seed for generating agent queries and simulated trajectories
- Use **Data Designer** to generate realistic multi-step user queries
- Simulate **agent trajectories** (step-by-step tool-call solutions)
- Apply **two levels of LLM judge filtering** to ensure data quality
- Export training data in **NeMo Gym format** for rollout collection and RLVR training

# 
#
> **Context Note:** The primary goal of this notebook is user query generation. The trajectory generation step in this notebook serves as a sanity check to ensure the generated query leads to a feasible path. In production RL training, rollout (oracle trajectory) traces are generated from the environment itself. You may find more information in [NeMo Gym Rollout Collection](https://docs.nvidia.com/nemo/gym/latest/get-started/rollout-collection.html) documentation.

## Architecture Overview

```
 ┌───────────────────────────────────────────────────────────────┐
 │                  DATA GENERATION PIPELINE                     │
 ├───────────────────────────────────────────────────────────────┤
 │                                                               │
 │   ┌──────────────┐    ┌──────────────┐    ┌──────────────┐    │
 │   │ Tool Schemas │───▶│  User Query  │───▶│  Trajectory  │    │
 │   │    (Seed)    │    │ Generation   │    │ Simulation   │    │
 │   └──────────────┘    └──────────────┘    └──────────────┘    │
 │                                                  │            │
 │                                                  ▼            │
 │                                           ┌──────────────┐    │
 │                                           │  LLM Judge   │    │
 │                                           │  (Quality)   │    │
 │                                           └──────────────┘    │
 │                                                  │            │
 │                                                  ▼            │
 │                                           ┌──────────────┐    │
 │                                           │  NeMo Gym    │    │
 │                                           │   Format     │    │
 │                                           └──────────────┘    │
 │                                                               │
 └───────────────────────────────────────────────────────────────┘
```

## Step 1: Import Dependencies

In [ ]:
# Verify dependencies are installed

import importlib

required_packages = ["data_designer", "pydantic", "pandas"]
missing_packages = [pkg for pkg in required_packages if importlib.util.find_spec(pkg) is None]

if missing_packages:
    missing_display = ", ".join(missing_packages)
    raise ImportError(
        "Missing required packages: "
        f"{missing_display}. Install them first with `uv pip install -r requirements.txt`."
    )

print("Dependencies available in the active notebook environment.")

In [ ]:
import json
from typing import List
from pydantic import BaseModel, Field
import pandas as pd

# Data Designer imports
from data_designer.config import (
    ChatCompletionInferenceParams,
    DataDesignerConfigBuilder,
    LLMStructuredColumnConfig,
    LLMTextColumnConfig,
    LocalFileSeedSource,
    ModelConfig,
    SamplingStrategy,
    ModelProvider,
)
from data_designer.interface import DataDesigner

## Context: What is the Workplace Assistant Environment?

[**Workplace Assistant**](https://docs.nvidia.com/nemo/gym/latest/tutorials/nemo-rl-grpo/about-workplace-assistant.html#) is a multi-step tool-using benchmark environment used in **NeMo Gym** for RL training. A model gets a natural language business request and must call tools in the right order with valid arguments (up to 6 steps).

At a high level:
- The model reads a user request (for example, scheduling meetings or updating CRM records)
- The model decides which tools to call and with what parameters
- The environment verifies correctness using **state matching** (final database state), not exact step matching

In this notebook, we focus on **data generation**: starting from known tool schemas, generating realistic user requests, and simulating feasible trajectories to produce NeMo Gym-compatible training data.

> **Note:** The official NeMo Gym [Workplace Assistant environment](https://github.com/NVIDIA-NeMo/Gym/tree/main/resources_servers/workplace_assistant) is the training target. This notebook is an example synthetic data preparation stage that feeds that workflow.

## Step 2: Load Tool Definitions

This notebook begins with **established tool schemas** and leverages them as foundational context for data generation. These schemas represent the (possibly domain-specific) tools on which you aim to enhance model performance.

We use 27 tools across 6 tool groups:
- **Company Directory**: Look up employee email addresses
- **Email**: Send, search, reply, forward, delete emails
- **Calendar**: Create, search, update, delete events
- **Analytics**: Query website visitor data and create plots
- **Project Management**: Manage tasks across Kanban boards
- **CRM**: Manage customer records and sales pipeline

These tools are designed to require **multi-step reasoning**. For example:
- "Email John about the meeting" requires first looking up John's email, then sending
- "Reassign all of Sarah's leads to Mike" requires looking up emails, searching customers, then updating each one

> **Why this matters:** The schemas define valid arguments and values (for example, allowed board/list/status values). We use these constraints to generate realistic user queries and schema-compliant simulated trajectories.

In [ ]:
# Load tool definitions from separate JSON files (one per database)
import os

TOOLS_DIR = 'tools'

# Load environment config
with open(os.path.join(TOOLS_DIR, 'environment.json'), 'r') as f:
    env_config = json.load(f)

SYSTEM_PROMPT = env_config['system_prompt']
MULTI_STEP_PATTERNS = env_config['common_multi_step_patterns']

# Load tools from each database file
DATABASE_FILES = [
    'company_directory.json',
    'email.json', 
    'calendar.json',
    'analytics.json',
    'project_management.json',
    'customer_relationship_manager.json'
]

TOOLS = []
DATABASES = {}
TOOL_CATEGORIES = {}

for db_file in DATABASE_FILES:
    with open(os.path.join(TOOLS_DIR, db_file), 'r') as f:
        db_config = json.load(f)
        
    db_name = db_config['database']
    DATABASES[db_name] = {
        'description': db_config['description'],
    }
    
    # Add tools and track category
    db_tools = db_config['tools']
    TOOLS.extend(db_tools)
    TOOL_CATEGORIES[db_name] = [t['name'] for t in db_tools]

print(f"Loaded {len(TOOLS)} tools across {len(DATABASES)} databases")
print(f"\nDatabases:")
for db_name, db_info in DATABASES.items():
    tool_count = len(TOOL_CATEGORIES[db_name])
    print(f"  - {db_name}: {tool_count} tools")
    print(f"    {db_info['description']}")

**`tools/*.json` Schema Format**

Each JSON file in `tools/` defines one database and its associated tools, emulating tool schemas a company might maintain:

| Field | Description |
|-------|-------------|
| `database` | Database name (e.g. `"calendar"`) |
| `description` | Human-readable description of the database |
| `data_schema` | Database structure — `columns` lists field names, `enums` (if present) defines constrained values. It is not currently consumed by the generation pipeline. |
| `tools` | Array of function definitions: `name`, `description`, `parameters` (JSON Schema), `operation_type` |

Only `description` and `tools` are used by the generation pipeline. `data_schema` is retained in the JSON files as realistic metadata context.

Display all loaded tools grouped by database. This summary shows each tool's name and description.

In [ ]:
# Helper function to format tools for prompts
def format_tools_for_prompt(tools: List[dict], include_schemas: bool = False) -> str:
    """Format tool definitions into a readable string for LLM prompts."""
    lines = []
    for tool in tools:
        lines.append(f"- **{tool['name']}**: {tool['description']}")
        if include_schemas:
            params = tool['parameters']['properties']
            if params:
                lines.append(f"  Parameters: {list(params.keys())}")
    return "\n".join(lines)

# Display tool summary by category
for category, tool_names in TOOL_CATEGORIES.items():
    print(f"\n### {category.upper()} ({len(tool_names)} tools)")
    category_tools = [t for t in TOOLS if t['name'] in tool_names]
    print(format_tools_for_prompt(category_tools))

## Step 3: Define Output Schemas

**Data Designer** uses **Pydantic** models to define structured output formats, ensuring the LLM generates data in a consistent, parseable format.

We define five schemas:
1. **ToolCall** / **AgentStep** / **AgentTrajectory**: Represent a multi-step tool-calling solution
2. **UserQueryJudgeScores**: Quality scores for generated user queries
3. **TrajectoryJudgeScores**: Quality scores for generated trajectories

In [ ]:
class ToolCall(BaseModel):
    """A single tool invocation."""
    name: str = Field(..., description="The name of the tool to call (e.g., 'email_send_email')")
    arguments: str = Field(..., description="JSON string of the tool arguments")


class AgentStep(BaseModel):
    """A single step in the agent's reasoning trajectory."""
    step_number: int = Field(..., description="The step number (1-indexed)")
    thought: str = Field(
        ..., 
        description="The agent's reasoning about what to do next and why. Should explain the purpose of the tool call."
    )
    tool_call: ToolCall = Field(..., description="The tool to call in this step")
    expected_result: str = Field(
        ..., 
        description="What information or state change we expect from this tool call"
    )


class AgentTrajectory(BaseModel):
    """Complete trajectory for solving a multi-step task."""
    reasoning_trace: List[AgentStep] = Field(
        ..., 
        description="The sequence of steps to solve the task. Should be 1-6 steps."
    )
    final_answer: str = Field(
        ..., 
        description="A brief confirmation of what was accomplished"
    )


class UserQueryJudgeScores(BaseModel):
    """Quality scores for the generated user query."""
    feasibility: int = Field(
        ..., ge=1, le=5, 
        description="Is the request achievable with the available tools? (1=impossible, 5=fully achievable)"
    )
    schema_compliance: int = Field(
        ..., ge=1, le=5, 
        description="Does the request use valid values as defined in tool schemas (e.g., valid board names, list names, statuses)? (1=uses invalid values, 5=all values valid)"
    )
    naturalness: int = Field(
        ..., ge=1, le=5, 
        description="Does the request sound like a natural user query? (1=robotic/artificial, 5=very natural)"
    )
    is_valid: bool = Field(
        ..., 
        description="True if the query is valid and should be kept, False if it should be discarded"
    )
    issues: str = Field(
        ..., 
        description="List any issues found (invalid enum values, impossible requests, etc.) or 'None' if valid"
    )


class TrajectoryJudgeScores(BaseModel):
    """Quality scores for the generated trajectory."""
    tool_validity: int = Field(
        ..., ge=1, le=5, 
        description="Are all tool names valid and arguments schema-compliant? (1=invalid tools/args, 5=all valid)"
    )
    argument_validity: int = Field(
        ..., ge=1, le=5, 
        description="Do all arguments use valid values as specified in tool descriptions? (1=invalid values, 5=all valid)"
    )
    completeness: int = Field(
        ..., ge=1, le=5, 
        description="Does the trajectory fully solve the user request? (1=incomplete, 5=fully complete)"
    )
    efficiency: int = Field(
        ..., ge=1, le=5, 
        description="Is the trajectory optimal without unnecessary steps? (1=very inefficient, 5=optimal)"
    )
    is_valid: bool = Field(
        ..., 
        description="True if the trajectory is valid and executable, False if it has errors"
    )
    issues: str = Field(
        ..., 
        description="List any issues found (invalid enum values, wrong tool names, missing steps, etc.) or 'None' if valid"
    )

## Step 4: Define Generation Prompts

The heart of synthetic data generation is the prompts. We define four prompts using **Jinja2 templates** (with `{{ variable }}` placeholders that Data Designer fills from seed columns):

1. **User Query Generation**: Create realistic workplace requests
2. **Trajectory Simulation**: Generate the step-by-step tool-call solution
3. **User Query Judge**: Evaluate query feasibility and schema compliance
4. **Trajectory Judge**: Evaluate tool-call correctness and completeness

### Key Principles
- **Specificity**: Tell the LLM exactly what format you want
- **Examples**: Show don't tell — include concrete examples by complexity level
- **Constraints**: Define what NOT to do (avoid trivial tasks, don't skip steps)

In [ ]:
# Prompt 1: Generate a realistic user query that may require one or more tool calls
USER_QUERY_GENERATION_PROMPT = """
You are creating training data for a workplace assistant AI agent.

**Your Task:** Generate a realistic user request that requires the agent to use one or more tools to complete.

**Available Tools (with full schemas):**
{{ tools_json }}

**Selected Tool Category:** {{ category }}

**Multi-Step Pattern to Use:** {{ pattern }}

**CRITICAL - Valid Values:**
Many tool parameters have RESTRICTED VALUES specified in their descriptions. You MUST only reference values that exist in the tool schemas. Pay close attention to phrases like "One of:" in parameter descriptions.

Common restrictions to follow:
- `list_name`: Only use 'Backlog', 'In Progress', 'In Review', or 'Completed' (NOT 'Prospects', 'Todo', 'Pipeline', etc.)
- `board`: Only use 'Back end', 'Front end', or 'Design' (NOT 'Sales', 'Marketing', 'Engineering', etc.)
- `status`: Only use 'Qualified', 'Won', 'Lost', 'Lead', or 'Proposal' (NOT 'Active', 'Prospect', 'Closed', etc.)
- `product_interest`: Only use 'Software', 'Hardware', 'Services', 'Consulting', or 'Training'

**Guidelines:**
1. The request should sound natural - like something a real employee would ask
2. It should require 1-6 tool calls to complete
3. Include specific details that make the task concrete (names, dates, subjects)
4. Don't mention tool names or technical details - speak like a normal user
5. The task MUST be achievable with the available tools using ONLY valid parameter values
6. When referencing boards, lists, statuses, etc., use EXACTLY the values allowed in the tool schemas

**Examples by Complexity:**

*Simple (1 step):*
- "Reply to Carlos's last email about the prototype with 'Thanks, I'll review it tomorrow'"
- "Change the name of my 3pm meeting to 'Risk Management Forum'"
- "How many website visitors did we have last week?"

*Medium (2-3 steps):*
- "Send an email to John about the quarterly review meeting tomorrow"
- "Schedule a 30-minute sync with Lisa tomorrow at 2pm"
- "Get the total visits and engaged users for November"

*Complex (4-6 steps):*
- "Raj is taking over all of Akira's leads that are interested in software. Can you reassign them in the CRM?"
- "Forward the last email from marketing about the Q4 report to everyone on the design team"
- "Move all of Sarah's overdue tasks on the Back end board to the Backlog"

**Output:** Return ONLY the user request as a single string. No quotes, no explanation.
"""

print("User Query Generation Prompt loaded")

In [ ]:
# Prompt 2: Simulate the agent's trajectory for solving the task
TRAJECTORY_SIMULATION_PROMPT = """
You are simulating an expert workplace assistant agent solving a task step-by-step.

**User Request:**
{{ user_query }}

**System Context:**
{{ system_prompt }}

**Available Tools:**
{{ tools_json }}

**Your Task:** Generate a step-by-step trajectory showing how the agent would solve this request.

**Guidelines:**
1. **Think Step-by-Step**: Each step should have a clear thought explaining WHY we're calling this tool
2. **Use Real Tool Names**: The tool_call.name must exactly match one of the available tools
3. **Valid JSON Arguments**: The tool_call.arguments must be valid JSON matching the tool's parameter schema
4. **Realistic IDs**: When referencing IDs discovered in previous steps, use placeholder format like "00000001"
5. **Complete the Task**: The trajectory must fully solve the user's request
6. **1-6 Steps**: Use the minimum number of steps needed. Simple tasks may need only 1 step.

**Common Patterns:**
- Look up a person's email before sending them a message
- Search for records before updating/deleting them
- Get information from one database to use in another
- Some tasks can be completed in a single step (e.g., reply to an email, update an event)

**Example Step:**
{% raw %}
```json
{
  "step_number": 1,
  "thought": "The user wants to email Raj, but I need his email address first. I'll look it up in the company directory.",
  "tool_call": {
    "name": "company_directory_find_email_address",
    "arguments": "{\"name\": \"Raj\"}"
  },
  "expected_result": "Raj's email address (likely raj.patel@atlas.com)"
}
```
{% endraw %}

Output the complete AgentTrajectory with all steps needed to solve the task.
"""

print("Trajectory Simulation Prompt loaded")

In [ ]:
# Prompt 3a: Judge the quality of the generated USER QUERY
USER_QUERY_JUDGE_PROMPT = """
You are a quality assurance judge evaluating a synthetically generated user query for training an AI workplace assistant.

**Generated User Query:**
{{ user_query }}

**Available Tools (with full schemas):**
{{ tools_json }}

**Your Task:** Evaluate whether this user query is valid and achievable with the available tools.

**CRITICAL - Check for Schema Compliance:**
Many tools have RESTRICTED VALUES for certain fields. The user query must only reference values that are valid according to the tool schemas. For example:
- If a tool says `list_name` must be one of 'Backlog', 'In Progress', 'In Review', 'Completed' - the query cannot ask for a "Prospects" list
- If a tool says `board` must be one of 'Back end', 'Front end', 'Design' - the query cannot ask for a "Sales" board  
- If a tool says `status` must be one of 'Qualified', 'Won', 'Lost', 'Lead', 'Proposal' - the query cannot use other statuses

**Evaluation Criteria:**

1. **Feasibility (1-5)**: Can this request be fulfilled using the available tools?
   - Score 1 if the request requires tools/capabilities that don't exist
   - Score 5 if the request is fully achievable with available tools

2. **Schema Compliance (1-5)**: Does the request use valid values?
   - Score 1 if the query references invalid enum values (wrong board names, list names, statuses, etc.)
   - Score 3 if the query is ambiguous but could map to valid values
   - Score 5 if all referenced values exactly match valid options in tool schemas

3. **Naturalness (1-5)**: Does this sound like a real user request?
   - Score 1 if robotic or artificial sounding
   - Score 5 if very natural and realistic

**is_valid:** Set to False if feasibility < 3 OR schema_compliance < 3. These queries should be discarded.

**issues:** List specific problems found. Examples:
- "References 'Sales' board but valid boards are: 'Back end', 'Front end', 'Design'"
- "References 'Prospects' list but valid lists are: 'Backlog', 'In Progress', 'In Review', 'Completed'"
- "None" if no issues found

**Output:** Return UserQueryJudgeScores with all fields.
"""

# Prompt 3b: Judge the quality of the generated TRAJECTORY
TRAJECTORY_JUDGE_PROMPT = """
You are a quality assurance judge evaluating a generated trajectory (sequence of tool calls) for training an AI workplace assistant.

**User Request:**
{{ user_query }}

**Generated Trajectory:**
{{ trajectory }}

**Available Tools (with full schemas):**
{{ tools_json }}

**Your Task:** Evaluate whether this trajectory correctly solves the user request using valid tool calls.

**CRITICAL - Check for Argument Validity:**
Tool arguments must use EXACTLY the values allowed by the tool schemas. For example:
- `list_name` must be one of: 'Backlog', 'In Progress', 'In Review', 'Completed' (NOT 'Prospects', 'Todo', etc.)
- `board` must be one of: 'Back end', 'Front end', 'Design' (NOT 'Sales', 'Marketing', etc.)
- `status` must be one of: 'Qualified', 'Won', 'Lost', 'Lead', 'Proposal' (NOT 'Active', 'Prospect', etc.)
- `product_interest` must be one of: 'Software', 'Hardware', 'Services', 'Consulting', 'Training'

**Evaluation Criteria:**

1. **Tool Validity (1-5)**: Are all tool names correct?
   - Score 1 if any tool name doesn't match available tools
   - Score 5 if all tool names exactly match

2. **Argument Validity (1-5)**: Are all arguments schema-compliant?
   - Score 1 if any argument uses invalid enum values or wrong types
   - Score 3 if arguments are mostly valid but some are ambiguous
   - Score 5 if all arguments perfectly match the schema requirements

3. **Completeness (1-5)**: Does the trajectory fully solve the request?
   - Score 1 if major parts of the request are unaddressed
   - Score 5 if the trajectory completely fulfills the request

4. **Efficiency (1-5)**: Is the trajectory optimal?
   - Score 1 if there are many unnecessary steps
   - Score 5 if the trajectory is optimal with no wasted steps

**is_valid:** Set to False if tool_validity < 4 OR argument_validity < 4. These trajectories have errors and should be discarded.

**issues:** List specific problems found. Examples:
- "Step 2 uses list_name='Prospects' but valid values are: 'Backlog', 'In Progress', 'In Review', 'Completed'"
- "Step 1 calls 'email_send' but correct tool name is 'email_send_email'"
- "None" if no issues found

**Output:** Return TrajectoryJudgeScores with all fields.
"""

print("User Query Judge Prompt loaded")
print("Trajectory Judge Prompt loaded")

## Step 5: Create Seed Data

**Data Designer** works by expanding seed data through LLM generation. Each seed row provides context variables that get substituted into the prompt templates:

- `category`: Which tool database to focus on (ensures diversity across domains)
- `pattern`: Which multi-step pattern to use (e.g., lookup-then-send, search-then-update)
- `tools_json`: Full tool schemas for the LLM to reference
- `system_prompt`: The system context for the workplace assistant

> **Pattern Engineering Note:** The multi-step patterns used as seeds in `create_seed_data()` are domain-informed. In practice, you can engineer these patterns from heuristics, inferred tool-call chains observed in production traffic, or other rule-based design choices. In this case, we had some common patterns stored in `tools/environments.json`.

In [ ]:
def create_seed_data(num_seeds: int = 100) -> pd.DataFrame:
    """
    Create seed data for the Data Designer pipeline.
    
    Each seed contains:
    - category: Which tool category to focus on
    - pattern: Which multi-step pattern to use
    - tools_description: Formatted tool descriptions
    - tools_json: Full tool schemas as JSON
    - system_prompt: The system context
    """
    seeds = []
    
    categories = list(TOOL_CATEGORIES.keys())
    patterns = [
        f"{p['pattern']}: {p['description']}" for p in MULTI_STEP_PATTERNS
    ]
    
    for i in range(num_seeds):
        # Select category and pattern (ensuring diversity)
        category = categories[i % len(categories)]
        pattern = patterns[i % len(patterns)]
        
        # Get tools for this category (plus company_directory for lookups)
        relevant_tool_names = TOOL_CATEGORIES[category] + TOOL_CATEGORIES.get('company_directory', [])
        relevant_tools = [t for t in TOOLS if t['name'] in relevant_tool_names]
        
        seeds.append({
            'seed_id': i,
            'category': category,
            'pattern': pattern,
            'tools_description': format_tools_for_prompt(relevant_tools, include_schemas=True),
            'tools_json': json.dumps(relevant_tools, indent=2),
            'tools_summary': format_tools_for_prompt(TOOLS),  # All tools for judge
            'system_prompt': SYSTEM_PROMPT,
        })
    
    return pd.DataFrame(seeds)

# Create seed data
seed_df = create_seed_data(num_seeds=50)
print(f"Created {len(seed_df)} seeds")
print(f"\nSample seed:")
print(seed_df.iloc[0].to_dict())

Save the seed data as a Parquet file for Data Designer to consume.

In [ ]:
# Save seeds to parquet for Data Designer
seed_df.to_parquet('workplace_assistant_seeds.parquet', index=False)
print("Seeds saved to workplace_assistant_seeds.parquet")

## Step 6: Configure the Data Designer Pipeline

Now we wire everything together into a **Data Designer** workflow:

1. **Load Seeds** — Provides category, pattern, and tools for each generation
2. **Generate User Query** — LLM creates a realistic workplace request
3. **Judge User Query** — LLM validates feasibility and schema compliance
4. **Simulate Trajectory** — LLM generates the step-by-step tool-call solution
5. **Judge Trajectory** — LLM validates tool names and argument correctness

### Configuration

First, set up the **NVIDIA Inference API** provider and model. If you do not already have a key, go to [build.nvidia.com/settings/api-keys](https://build.nvidia.com/settings/api-keys), sign in, and create one. The next cell will prompt you for the value if `NVIDIA_API_KEY` is not already set in your environment.

In [ ]:
import getpass

if "NVIDIA_API_KEY" not in os.environ or not os.environ["NVIDIA_API_KEY"]:
    os.environ["NVIDIA_API_KEY"] = getpass.getpass("Enter your NVIDIA API key: ")

In [ ]:
# Define custom provider pointing to NVIDIA build.nvidia.com API
NVIDIA_INFERENCE_URL = "https://integrate.api.nvidia.com/v1"

custom_providers = [
    ModelProvider(
        name="nvidia-inference",
        endpoint=NVIDIA_INFERENCE_URL,
        provider_type="openai",
        api_key=os.environ.get("NVIDIA_API_KEY", ""),
    ),
]

# Model name must match NVIDIA's model identifier
MODEL_ID = "openai/gpt-oss-120b"
MODEL_ALIAS = "gpt-oss-120b"

model_configs = [
    ModelConfig(
        alias=MODEL_ALIAS,
        model=MODEL_ID,
        provider="nvidia-inference",
        inference_parameters=ChatCompletionInferenceParams(
            max_tokens=16384,
        ),
    )
]

# Initialize DataDesigner and config builder
data_designer = DataDesigner(model_providers=custom_providers)
config_builder = DataDesignerConfigBuilder(model_configs=model_configs)

Build the pipeline with four generation columns: user query, user query judge, trajectory, and trajectory judge.

In [ ]:
def build_workplace_assistant_pipeline():
    """
    Build the complete Data Designer pipeline for generating 
    multi-step tool-calling training data.
    
    Pipeline stages:
    1. Generate user query
    2. Judge user query (filter invalid queries early)
    3. Generate trajectory 
    4. Judge trajectory (filter invalid trajectories)
    """
    
    # Initialize the config builder
    config_builder = DataDesignerConfigBuilder(model_configs=model_configs)
    
    # Load seed data
    seed_ref = LocalFileSeedSource(path='workplace_assistant_seeds.parquet')
    config_builder.with_seed_dataset(seed_ref, sampling_strategy=SamplingStrategy.SHUFFLE)
    
    # Column 1: Generate User Query
    # This creates a realistic workplace request based on the category and pattern
    config_builder.add_column(
        LLMTextColumnConfig(
            name="user_query",
            prompt=USER_QUERY_GENERATION_PROMPT,
            model_alias=MODEL_ALIAS,
        )
    )
    
    # Column 2: Judge User Query
    # Validates that the user query is feasible and uses valid enum values
    config_builder.add_column(
        LLMStructuredColumnConfig(
            name="user_query_judge",
            prompt=USER_QUERY_JUDGE_PROMPT,
            output_format=UserQueryJudgeScores,
            model_alias=MODEL_ALIAS,
        )
    )
    
    # Column 3: Simulate Agent Trajectory
    # This generates the step-by-step solution with tool calls
    config_builder.add_column(
        LLMStructuredColumnConfig(
            name="trajectory",
            prompt=TRAJECTORY_SIMULATION_PROMPT,
            output_format=AgentTrajectory,
            model_alias=MODEL_ALIAS,
        )
    )
    
    # Column 4: Judge Trajectory
    # Validates that the trajectory uses correct tool names and valid argument values
    config_builder.add_column(
        LLMStructuredColumnConfig(
            name="trajectory_judge",
            prompt=TRAJECTORY_JUDGE_PROMPT,
            output_format=TrajectoryJudgeScores,
            model_alias=MODEL_ALIAS,
        )
    )
    
    return config_builder

# Build the pipeline
pipeline = build_workplace_assistant_pipeline()
print("Pipeline configured with 4 generation columns:")
print("  1. user_query (text) - Generate realistic user request")
print("  2. user_query_judge (structured) - Validate query feasibility and schema compliance")
print("  3. trajectory (structured) - Generate step-by-step solution")
print("  4. trajectory_judge (structured) - Validate tool calls and argument values")

Validate the pipeline configuration to catch any issues before generation.

In [ ]:
data_designer.validate(pipeline)

Run a quick preview with 2 records to verify the pipeline produces well-formed outputs before scaling up.

This step can take up to 2 minutes to complete.

In [ ]:
preview = data_designer.preview(pipeline, num_records=2)

Inspect a sample generated user query from the preview.

In [ ]:
preview.dataset

In [ ]:
preview.dataset.user_query[0], preview.dataset.trajectory[0]

## Step 7: Set Up Quality Filtering

We  now apply two levels of quality filtering to keep only high-quality examples.

This stage is generic to synthetic data generation and not specific to NeMo Gym.

To keep this notebook clean, quality filtering helpers live in `utils/quality_filtering.py`.

In [ ]:
from utils import filter_high_quality, show_rejection_reasons

## Step 8: Generate and Filter the Dataset

Run the full pipeline end-to-end: generate records and apply two levels of **quality filtering**.

```
┌──────────────┐    ┌──────────────┐    ┌──────────────┐    ┌──────────────┐
│   Generate   │───▶│ Stage 1:     │───▶│ Stage 2:     │───▶│   Filtered   │
│   Records    │    │ Query Judge  │    │ Traj Judge   │    │   Dataset    │
└──────────────┘    └──────────────┘    └──────────────┘    └──────────────┘
```

**Utility location:** `utils/quality_filtering.py`

**Quick usage:**
- Run `show_rejection_reasons(results_df, num_examples=3)` to inspect failures
- Run `filter_high_quality(results_df, verbose=True)` to apply default strict filtering
- Optionally tune thresholds with `FilterThresholds(...).to_kwargs()`

**Why Two Levels of Quality Filtering?**
- **Stage 1 (User Query)**: Catches queries which are intractable in this context. Filters on the `UserQueryJudgeScores` fields (`feasibility`, `schema_compliance`, `naturalness`) produced by the LLM judge prompts defined in Step 4.
- **Stage 2 (Trajectory)**: Catches tool argument errors that slipped through, or that doesn't answer the query. Filters on the `TrajectoryJudgeScores` fields (`tool_validity`, `argument_validity`, `completeness`, `efficiency`) from Step 4.



The following step can take up to 5 minutes to complete.

In [ ]:
print("Generating 10 examples...")
results = data_designer.create(pipeline, num_records=10)

results_df = results.load_dataset()
print(f"\nGenerated {len(results_df)} records")
print("\nColumns:", list(results_df.columns))

Inspect rejection reasons at both judge levels to understand what kinds of errors the pipeline catches.

In [ ]:
# Show rows the LLM judge already marked is_valid=False before stricter threshold filtering below.
show_rejection_reasons(results_df, num_examples=10)

Apply two levels of quality filtering with strict schema compliance requirements. Records must pass **both** the user query judge and the trajectory judge to be kept.

In [ ]:
filtered_df = filter_high_quality(
    results_df,
    min_query_feasibility=3,
    min_query_schema_compliance=4,
    min_query_naturalness=3,
    min_trajectory_tool_validity=4,
    min_trajectory_argument_validity=4,
    min_trajectory_completeness=3,
    min_trajectory_efficiency=3,
    verbose=True,
)

filtered_df.head()

## Step 9: Convert to NeMo Gym Format and Save

Convert filtered records into NeMo Gym JSONL format and save them.

In [ ]:
from typing import Any


from functools import partial
import json
from utils import convert_to_nemo_gym_format, save_for_nemo_gym

convert_fn = partial[dict[str, Any]](convert_to_nemo_gym_format, tools=TOOLS, system_prompt=SYSTEM_PROMPT)

output_path = "workplace_assistant_train-gpt-oss.jsonl"
save_for_nemo_gym(filtered_df, output_path, convert_fn=convert_fn)


# Display a sample of the saved task data
with open(output_path) as f:
    first_row = json.loads(f.readline())

print("\nSample saved row:")
print(json.dumps(first_row, indent=2))

## Summary

This notebook demonstrated how to build a complete synthetic data generation pipeline for multi-step tool-calling tasks using **Data Designer**. The pipeline generates user queries, simulates agent trajectories, and applies two levels of LLM judge filtering to produce high-quality training data.

## Next Steps

- **Scale up generation**: Increase `num_seeds` and `num_records` to produce larger training sets (1,000+ examples)
- **Customize for your domain**: Replace the Workplace Assistant tools with your own tool definitions
- **Add more multi-step patterns**: Define new patterns in `environment.json` to increase task diversity
- **Tune judge thresholds**: Inspect rejected examples with `show_rejection_reasons()` and adjust filtering thresholds
- **Train with NeMo Gym and NeMo RL**: Use the exported JSONL file for GRPO training with NeMo RL, using a NeMo Gym environment.